# pyironflow

[`pyironflow`](https://github.com/pyiron/pyironFlow) is a visual programming environment for [`pyiron_workflow`](https://pyiron-workflow.readthedocs.io/en/latest/): it renders a workflow graph as boxes and wires in an interactive, drag-and-drop canvas, and lets you build, inspect and run one the same way, without writing the underlying Python. Where `pyiron_workflow` itself is aimed at users who are comfortable declaring nodes and wiring them up in code, `pyironflow` puts a graphical user interface in front of that same graph, targeting domain scientists and other less technically experienced users who want to compose and run a workflow without programming.

This notebook rebuilds the same [`workflow.py`](workflow.py) analysis from [`python.ipynb`](python.ipynb) — read CSV → convert load to stress/strain → fit Young's modulus → plot — as a `pyiron_workflow` graph, exactly as in [`pyiron_workflow.ipynb`](pyiron_workflow.ipynb), and then hands that graph to `pyironflow` to show what the same workflow looks like, and can be operated, through the GUI.

In [1]:
from concurrent.futures import Future
from pyiron_workflow import Workflow, to_function_node
from pyironflow import PyironFlow
from workflow import (
    read_csv as _read_csv, 
    calculate_youngs_modulus as _calculate_youngs_modulus, 
    convert_load_to_stress as _convert_load_to_stress, 
    plot as _plot,
)

## Nodes

`to_function_node` wraps an existing Python function — unchanged — into a `pyiron_workflow` node. Each of the function's arguments becomes a node input, and its return value(s) become node output(s); nothing about `read_csv`, `convert_load_to_stress`, `calculate_youngs_modulus` or `plot` themselves had to change to make this work.

In [2]:
read_csv = to_function_node("read_csv", _read_csv, "read_csv")
calculate_youngs_modulus = to_function_node("calculate_youngs_modulus", _calculate_youngs_modulus, "calculate_youngs_modulus")
convert_load_to_stress = to_function_node("convert_load_to_stress", _convert_load_to_stress, "convert_load_to_stress")
plot = to_function_node("plot", _plot, "plot")

## Workflow

A `Workflow` is a container for nodes. Assigning `wf.<name> = node(...)` both adds the node to the graph and wires its inputs — either to a plain value or to another node's output, e.g. `wf.result_dict["stress"]`. Execution is lazy: nothing actually runs until `wf.run()` is called, at which point `pyiron_workflow` resolves the dependencies and executes the nodes in the right order.

This is the one part of the notebook still written in code — normally the point of `pyironflow` is that a user builds exactly this graph by dragging nodes onto the canvas instead.

In [3]:
wf = Workflow("my_workflow")

In [4]:
wf.area = 120
wf.strain_cutoff = 0.2
wf.filename = "./data/dataset_1.csv"
wf.df = read_csv(filename=wf.filename)
wf.result_dict = convert_load_to_stress(df=wf.df, area=wf.area)
wf.youngs_modulus = calculate_youngs_modulus(stress=wf.result_dict["stress"], strain=wf.result_dict["strain"], strain_cutoff=wf.strain_cutoff)
wf.plot = plot(stress=wf.result_dict["stress"], strain=wf.result_dict["strain"], format="-")

## Visual Editor

`PyironFlow(wf_list=[wf], root_path=".")` opens `wf` in the visual editor: each node becomes a box showing its inputs and outputs, connected by the same edges declared above, and `root_path` tells the editor where to look for workflow files to browse and load. From here the graph can be inspected, edited and re-run entirely by clicking and dragging — the same workflow, but without touching the code that built it.

In [5]:
pf = PyironFlow(wf_list=[wf], root_path=".")
pf.gui

![image](images/vp_constructed.png)

## Reload Workflow

Opening `PyironFlow` with an empty `Workflow("workflow_reloaded")` shows the other side of the same idea: a user who has never seen `workflow.py` can still browse to a saved workflow file under `root_path` and load it into the canvas from the GUI alone, then inspect, rerun or adapt it — no Python required at any point.

In [6]:
pf = PyironFlow(wf_list=[Workflow("workflow_reloaded")], root_path=".")
pf.gui

![image](images/vp_loading.png)

## Summary

This is the same `pyiron_workflow` graph built in [`pyiron_workflow.ipynb`](pyiron_workflow.ipynb), now composed, inspected and reloaded entirely through `pyironflow`'s graphical canvas — no Python required. See [`python.ipynb`](python.ipynb) for where the underlying `read_csv`, `convert_load_to_stress`, `calculate_youngs_modulus` and `plot` functions come from, or [`executorlib.ipynb`](executorlib.ipynb) for the same functions scaled up with a lightweight, HPC-ready executor.